# Run bike-lane detection on Google Colab

This notebook runs the whole Drive-based detection pipeline for **both** cameras.

Your recorded clips live on **Google Drive** and are named like
`commave_<camID>_YYYY-MM-DD_HHMM.mp4` (for example `commave_camA_2024-05-01_0830.mp4`).

Two cameras film the same street from different angles. **camA and camB are processed
separately and their detections/counts are never combined** (Requirement 12.5). That is
why there is a separate detection cell for each camera below.

What this notebook does, in order:
1. Mount Google Drive.
2. Get the code (clone or copy the repo) and `cd` into it.
3. Install dependencies.
4. Set your Drive paths.
5. Export a first frame per camera (so you can pick points on your laptop) **before** detection.
6. Run detection for **camA**.
7. Run detection for **camB** (in its own cell).

Edit the clearly-marked placeholders, then run the cells top to bottom.

In [ ]:
# Step 1: Mount Google Drive so Colab can read your videos and write CSVs back to Drive.
from google.colab import drive
drive.mount('/content/drive')

## Get the code

Next we bring the `bike-lane-project` repository into the Colab runtime so we can run
`python -m src.detect` and `python -m tools.export_frame`.

The cell below shows the **git clone** approach. If your repo is private or you keep a copy
of the code on Drive, you can copy it from Drive instead (see the commented alternative).
Either way we finish by `%cd`-ing into the repo folder so later cells run from the repo root.

In [ ]:
# Step 2: Get the code into the Colab runtime, then cd into the repo root.

# --- Option A: clone from GitHub (edit REPO_URL to your repo) ---
REPO_URL = "https://github.com/<your-org>/bike-lane-project.git"  # <-- EDIT THIS
!git clone $REPO_URL /content/bike-lane-project

# --- Option B (alternative): copy a repo you already keep on Drive ---
# If you prefer, comment out the git clone above and copy from Drive instead, e.g.:
# !cp -r "/content/drive/MyDrive/bike-lane/bike-lane-project" /content/bike-lane-project

# cd into the repo root so `python -m src.detect` and `src` imports resolve.
%cd /content/bike-lane-project

In [ ]:
# Step 3: Install the pinned dependencies (ultralytics, opencv, pandas, ...).
!pip install -r requirements.txt

## Set your Drive paths

Edit the variables below so they point at **your** Drive folders. The raw-video folders
should each contain the `commave_<camID>_..._.mp4` clips for that one camera. The output
and frames folders are created on Drive if they do not exist yet.

In [ ]:
# Step 4: Define your Drive paths. EDIT these placeholders to match your Drive layout.

# Raw video folders (each holds the commave_<camID>_..._.mp4 clips for that camera):
RAW_CAMA = "/content/drive/MyDrive/bike-lane/raw_video/camA"
RAW_CAMB = "/content/drive/MyDrive/bike-lane/raw_video/camB"

# Where the detections CSVs get written (kept separate per camera, never combined):
OUT_CAMA = "/content/drive/MyDrive/bike-lane/detections/camA"
OUT_CAMB = "/content/drive/MyDrive/bike-lane/detections/camB"

# Where exported first-frame PNGs go (download these to pick points on your laptop):
FRAMES_OUT = "/content/drive/MyDrive/bike-lane/frames"

# Per-camera config files live here (config/camA.yaml, config/camB.yaml):
CONFIG_DIR = "config"

## Export a first frame to pick points on your laptop

Before running detection, export one first-frame PNG per camera. Download the PNG from Drive
and run `tools/pick_points.py` **locally** on your laptop to click the counting line and lane
polygon, then paste the resulting YAML into that camera's config file.

This export step runs **before** detection (Requirement 12.3).

In [ ]:
# Step 5: Export one first frame per camera BEFORE detection (Req 12.3).
# We glob the first .mp4 in each raw folder and export its first frame as a PNG.
import glob
import os

def export_first_frame(raw_folder, frames_out):
    """Find the first .mp4 in raw_folder and export its first frame to frames_out."""
    clips = sorted(glob.glob(os.path.join(raw_folder, '*.mp4')))
    if not clips:
        print(f"No .mp4 clips found in {raw_folder} - skipping frame export.")
        return
    first_clip = clips[0]
    print(f"Exporting first frame of: {first_clip}")
    # $FRAMES_OUT names a folder, so export_frame writes <Video_Name>.png into it.
    os.system(f'python -m tools.export_frame --video "{first_clip}" --output "{frames_out}"')

export_first_frame(RAW_CAMA, FRAMES_OUT)  # one frame for camA
export_first_frame(RAW_CAMB, FRAMES_OUT)  # one frame for camB

## Run detection — Camera A

This cell runs detection for **camA only**. It reads the camA clips from Drive and writes
camA's detections CSVs back to Drive.

In [ ]:
# Step 6: Run detection for CAMERA A only (Req 12.4).
# Optional flags you can add: --stride N (process every Nth frame),
#   --max-seconds N (only the first N seconds), --save-annotated (write a blurred demo video).
# Example: !python -m src.detect --input "$RAW_CAMA" --output "$OUT_CAMA" --config-dir "$CONFIG_DIR" --stride 5 --max-seconds 30
!python -m src.detect --input "$RAW_CAMA" --output "$OUT_CAMA" --config-dir "$CONFIG_DIR"

## Run detection — Camera B

This cell runs detection for **camB only**, in a separate cell from camA. Keeping the two
runs in distinct cells is deliberate: camA and camB detections and counts are **never
combined** (Requirement 12.5).

In [ ]:
# Step 7: Run detection for CAMERA B only (Req 12.4), in its own cell.
# Same optional flags apply here (--stride, --max-seconds, --save-annotated).
!python -m src.detect --input "$RAW_CAMB" --output "$OUT_CAMB" --config-dir "$CONFIG_DIR"

## Done

Your detections CSVs are now on Google Drive:
- **camA** CSVs under `OUT_CAMA`
- **camB** CSVs under `OUT_CAMB`

These per-camera detections CSVs are what the rest of the pipeline reads.

**Reminder:** never combine camA and camB counts. The two cameras film the same street from
different angles, so adding their counts together would double-count the same traffic. Keep
each camera's numbers separate throughout the analysis (Requirement 12.5).